# Lineborn — Strict Pre-Release Colab Runtime

This notebook provides a temporary **Lineborn-compatible remote AI runtime** for pre-release testing. Direct SIP, campaigns, DNC, call state, transcripts, deterministic safety logic and carrier control stay inside the Lineborn desktop app. Colab runs only **Whisper → Qwen → Chatterbox**.

### Before you start
1. Choose **Runtime → Change runtime type → GPU** in Colab.
2. Run **Cell 1** and wait for `LINEBORN COLAB RUNTIME READY`. First setup can take a while because the AI stack and models are installed.
3. Copy the printed **Runtime URL**, **Pairing token**, and exact **Model** into Lineborn → Settings → AI Compute.
4. In Lineborn, save/test the connection, choose a valid voice reference and click **Start AI**. This uploads the voice and warms the remote stack.
5. Return here and run **Cell 2** for the strict pre-release compute benchmark.

The benchmark is intentionally strict. A failure is useful: do not loosen thresholds just to produce a green result. Colab availability and session duration are controlled by Google; this notebook is benchmark infrastructure, not production hosting.


In [ ]:
# Cell 1 — Start the Lineborn runtime and obtain pairing credentials
import json, os, pathlib, shutil, subprocess, sys, time

ROOT = pathlib.Path('/content')
REPO = ROOT / 'Axemetric-Caller-Beta-Runtime'
PAIRING = ROOT / 'lineborn-colab-pairing.json'
BOOT_LOG = ROOT / 'lineborn-colab-bootstrap.log'

gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('No NVIDIA GPU detected. Choose Runtime > Change runtime type > GPU, reconnect, then run this cell again.')
print('GPU(s):\n' + gpu.stdout.strip())

if REPO.exists():
    subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','--depth','1','https://github.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime.git',str(REPO)], check=True)
sha = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
print('Lineborn runtime source:', sha)

PAIRING.unlink(missing_ok=True)
BOOT_LOG.unlink(missing_ok=True)
bootstrap = REPO / 'colab' / 'lineborn_colab_bootstrap.py'
LINEBORN_LOG_HANDLE = BOOT_LOG.open('w', encoding='utf-8', buffering=1)
LINEBORN_PROC = subprocess.Popen([sys.executable, str(bootstrap), '--model', 'auto'], cwd=str(REPO), stdout=LINEBORN_LOG_HANDLE, stderr=subprocess.STDOUT, text=True, start_new_session=True)

deadline = time.time() + 2400
offset = 0
while time.time() < deadline:
    time.sleep(2)
    if BOOT_LOG.exists():
        text = BOOT_LOG.read_text('utf-8', errors='replace')
        if len(text) > offset:
            print(text[offset:], end='')
            offset = len(text)
    if PAIRING.exists():
        break
    if LINEBORN_PROC.poll() is not None:
        raise RuntimeError(f'Lineborn runtime exited with code {LINEBORN_PROC.returncode}. See {BOOT_LOG}.')
else:
    raise TimeoutError('Lineborn runtime did not become ready within 40 minutes.')

LINEBORN_PAIRING = json.loads(PAIRING.read_text('utf-8'))
print('\n' + '=' * 76)
print('LINEBORN COLAB RUNTIME READY')
print('=' * 76)
print('Runtime URL   :', LINEBORN_PAIRING['url'])
print('Pairing token :', LINEBORN_PAIRING['token'])
print('Model         :', LINEBORN_PAIRING['model'])
print('GPU           :', LINEBORN_PAIRING['gpu'], f"({LINEBORN_PAIRING['vram_mb']} MiB)")
print('\nNow pair Lineborn, upload/select your voice reference, and click Start AI. Then run Cell 2.')


In [ ]:
# Cell 2 — Strict pre-release compute benchmark
import json, pathlib, requests, time
from google.colab import files

ROOT = pathlib.Path('/content')
PAIRING = ROOT / 'lineborn-colab-pairing.json'
if not PAIRING.exists():
    raise RuntimeError('Run Cell 1 first.')
pairing = json.loads(PAIRING.read_text('utf-8'))
headers = {'Authorization': f"Bearer {pairing['token']}"}
local = pairing['local_url']
health = requests.get(local + '/lineborn/health', headers=headers, timeout=20).json()
print('Runtime health:', json.dumps(health, indent=2))
if not health.get('voice_ready'):
    raise RuntimeError('The voice is not ready. In Lineborn, select the Colab runtime, choose a valid voice reference, then click Start AI. When Lineborn shows the remote AI as ready, rerun this cell.')

print('\nRunning 7 strict warmed pipeline trials. Do not interact with the runtime while this runs...')
started = time.time()
response = requests.post(local + '/lineborn/benchmark', headers=headers, json={'iterations': 7}, timeout=1800)
if response.status_code >= 400:
    raise RuntimeError(f'Benchmark HTTP {response.status_code}: {response.text[-2000:]}')
report = response.json()
elapsed = time.time() - started
out = ROOT / 'lineborn-prerelease-benchmark.json'
out.write_text(json.dumps(report, indent=2), encoding='utf-8')

print('\n' + '=' * 76)
print('STRICT PRE-RELEASE RESULT:', 'PASS ✅' if report.get('strict_pass') else 'FAIL ❌')
print('=' * 76)
print('Model:', report.get('model'))
print('GPU:', report.get('gpu', {}).get('name'))
print('Benchmark wall time: %.1f s' % elapsed)
print('\nMETRICS')
for key, value in report.get('metrics', {}).items():
    print(f'  {key}: {value}')
print('\nGATES')
for key, value in report.get('gate_results', {}).items():
    print(f"  {'PASS' if value else 'FAIL'}  {key}")
if report.get('errors'):
    print('\nERRORS')
    for item in report['errors']:
        print(' ', item)
print('\nReport saved to:', out)
files.download(str(out))


## What this benchmark does and does not certify

The strict Colab gate measures the warmed remote **Whisper → Qwen → Chatterbox** path, including STT accuracy, LLM time-to-first-token, TTS real-time factor, composite voice-start latency, error-free repeated execution and GPU memory use.

A green result is **not** the entire Lineborn release gate. Direct SIP/PSTN behavior, carrier disconnects, DNC/tool routing, installer behavior, Windows/macOS integration, real network jitter and long-duration campaign soak testing still need their own checks. Do not treat a Colab PASS as proof that every production failure mode is solved.


In [ ]:
# Cell 3 — Stop the temporary runtime when testing is finished
import os, signal
try:
    if LINEBORN_PROC.poll() is None:
        os.killpg(os.getpgid(LINEBORN_PROC.pid), signal.SIGTERM)
        print('Lineborn Colab runtime stopped.')
    else:
        print('Runtime is already stopped.')
except NameError:
    print('No runtime process was started in this notebook session.')
